In [1]:
# ========================
# KDTree NODE SNAPPING + RISK-AWARE
# ========================

import json, time, numpy as np
from scipy.spatial import KDTree
import networkx as nx
import random

start_time = time.time()

with open("tokyo_graph.json") as f:
    graph = json.load(f)

print("Graph loaded")

nodes = graph["nodes"]
edges = graph["edges"]

# ========================
# SPEED + TRAVEL TIME
# ========================
SPEEDS = {"highway":25,"major":16,"minor":11,"local":7}

for e in edges:
    rc = e.get("road_class","local")
    e["speed"] = SPEEDS.get(rc,7)
    e["travel_time"] = e["length"] / e["speed"]

# ========================
# ADD RISK (BRIDGES + RIVERS)
# ========================
for e in edges:
    risk = 1.0

    if e.get("is_bridge"):
        risk *= 1.5

    if e.get("near_river"):
        risk *= 1.3

    e["risk_weight"] = e["travel_time"] * risk

print("Risk weights added")

# ========================
# SPLIT NODES
# ========================
road_nodes = [n for n in nodes if n.get("type") in ["road_node", None]]
poi_nodes = [n for n in nodes if n.get("type") in ["building","hospital","fire_station"]]

# ========================
# KD TREE
# ========================
tree = KDTree([(n["lat"], n["lon"]) for n in road_nodes])

new_edges = []
distances = []

# ========================
# CONNECT (k=3)
# ========================
for i, poi in enumerate(poi_nodes):

    if i % 500 == 0:
        print(f"{i}/{len(poi_nodes)}")

    dists, idxs = tree.query((poi["lat"], poi["lon"]), k=3)

    for d, idx in zip(dists, idxs):
        road = road_nodes[idx]
        length = d * 100000

        distances.append(length)

        risk_time = length / 5  # walking/entry speed

        new_edges.append({
            "from": poi["id"],
            "to": road["id"],
            "length": length,
            "travel_time": risk_time,
            "risk_weight": risk_time
        })

        new_edges.append({
            "from": road["id"],
            "to": poi["id"],
            "length": length,
            "travel_time": risk_time,
            "risk_weight": risk_time
        })

graph["edges"].extend(new_edges)

# ========================
# VALIDATION
# ========================
G = nx.Graph()
for e in graph["edges"]:
    G.add_edge(e["from"], e["to"], weight=e["risk_weight"])

components = list(nx.connected_components(G))

print("\n=== NODE KDTree METRICS ===")
print("Connections:", len(new_edges))
print("Avg dist:", sum(distances)/len(distances))
print("Components:", len(components))

# --- PATH TEST ---
nodes_list = list(G.nodes)
s, t = random.choice(nodes_list), random.choice(nodes_list)

try:
    path = nx.shortest_path(G, s, t, weight="weight")
    print("Sample path length:", len(path))
except:
    print("No path found")

# ========================
# SAVE
# ========================
with open("kdtree_node.json","w") as f:
    json.dump(graph,f)

print("Saved: kdtree_node.json")
print("Time:", time.time() - start_time)

Graph loaded
Risk weights added
0/10482
500/10482
1000/10482
1500/10482
2000/10482
2500/10482
3000/10482
3500/10482
4000/10482
4500/10482
5000/10482
5500/10482
6000/10482
6500/10482
7000/10482
7500/10482
8000/10482
8500/10482
9000/10482
9500/10482
10000/10482

=== NODE KDTree METRICS ===
Connections: 62892
Avg dist: 79.18330657733962
Components: 1
Sample path length: 101
Saved: kdtree_node.json
Time: 9.189377784729004


In [2]:
# ========================
# KDTree EDGE SNAPPING(APPROX) + RISK-AWARE
# ========================

import json, time, numpy as np
from scipy.spatial import KDTree
from collections import defaultdict
import networkx as nx
import random

start_time = time.time()

with open("tokyo_graph.json") as f:
    graph = json.load(f)

print("Graph loaded")

nodes = graph["nodes"]
edges = graph["edges"]

node_dict = {n["id"]: n for n in nodes}

# ========================
# SPEED + TRAVEL TIME
# ========================
SPEEDS = {"highway":25,"major":16,"minor":11,"local":7}

for e in edges:
    rc = e.get("road_class","local")
    e["speed"] = SPEEDS.get(rc,7)
    e["travel_time"] = e["length"] / e["speed"]

# ========================
# ADD RISK
# ========================
for e in edges:
    risk = 1.0

    if e.get("is_bridge"):
        risk *= 1.5

    if e.get("near_river"):
        risk *= 1.3

    e["risk_weight"] = e["travel_time"] * risk

print("Risk weights added")

# ========================
# SPLIT
# ========================
road_nodes = [n for n in nodes if n.get("type") in ["road_node", None]]
poi_nodes = [n for n in nodes if n.get("type") in ["building","hospital","fire_station"]]

# ========================
# KD TREE
# ========================
tree = KDTree([(n["lat"], n["lon"]) for n in road_nodes])

# ========================
# ADJACENCY
# ========================
adj = defaultdict(list)
for e in edges:
    adj[e["from"]].append(e)
    adj[e["to"]].append(e)

# ========================
# DIST FUNC
# ========================
def point_to_segment(px, py, x1, y1, x2, y2):
    dx, dy = x2-x1, y2-y1
    if dx == 0 and dy == 0:
        return ((px-x1)**2 + (py-y1)**2)**0.5
    t = ((px-x1)*dx + (py-y1)*dy) / (dx*dx + dy*dy)
    t = max(0, min(1, t))
    proj_x = x1 + t*dx
    proj_y = y1 + t*dy
    return ((px-proj_x)**2 + (py-proj_y)**2)**0.5

# ========================
# CONNECT
# ========================
new_edges = []
distances = []

for i, poi in enumerate(poi_nodes):

    if i % 500 == 0:
        print(f"{i}/{len(poi_nodes)}")

    _, idxs = tree.query((poi["lat"], poi["lon"]), k=3)

    candidate_edges = set()

    for idx in idxs:
        node = road_nodes[idx]
        for e in adj[node["id"]]:
            candidate_edges.add((e["from"], e["to"]))

    px, py = poi["lon"], poi["lat"]

    edge_dists = []

    for u,v in candidate_edges:
        n1, n2 = node_dict[u], node_dict[v]
        d = point_to_segment(px, py, n1["lon"], n1["lat"], n2["lon"], n2["lat"])
        edge_dists.append((d,u,v))

    edge_dists.sort(key=lambda x: x[0])

    for d,u,v in edge_dists[:3]:

        length = d * 100000
        risk_time = length / 5

        distances.append(length)

        new_edges.append({"from":poi["id"],"to":u,"length":length,"travel_time":risk_time,"risk_weight":risk_time})
        new_edges.append({"from":poi["id"],"to":v,"length":length,"travel_time":risk_time,"risk_weight":risk_time})

        new_edges.append({"from":u,"to":poi["id"],"length":length,"travel_time":risk_time,"risk_weight":risk_time})
        new_edges.append({"from":v,"to":poi["id"],"length":length,"travel_time":risk_time,"risk_weight":risk_time})

graph["edges"].extend(new_edges)

# ========================
# VALIDATION
# ========================
G = nx.Graph()
for e in graph["edges"]:
    G.add_edge(e["from"], e["to"], weight=e["risk_weight"])

components = list(nx.connected_components(G))

print("\n=== EDGE KDTree METRICS ===")
print("Connections:", len(new_edges))
print("Avg dist:", sum(distances)/len(distances))
print("Components:", len(components))

# --- PATH TEST ---
nodes_list = list(G.nodes)
s, t = random.choice(nodes_list), random.choice(nodes_list)

try:
    path = nx.shortest_path(G, s, t, weight="weight")
    print("Sample path length:", len(path))
except:
    print("No path found")

# ========================
# SAVE
# ========================
with open("kdtree_edge.json","w") as f:
    json.dump(graph,f)

print("Saved: kdtree_edge.json")
print("Time:", time.time() - start_time)

Graph loaded
Risk weights added
0/10482
500/10482
1000/10482
1500/10482
2000/10482
2500/10482
3000/10482
3500/10482
4000/10482
4500/10482
5000/10482
5500/10482
6000/10482
6500/10482
7000/10482
7500/10482
8000/10482
8500/10482
9000/10482
9500/10482
10000/10482

=== EDGE KDTree METRICS ===
Connections: 125784
Avg dist: 25.685237460559698
Components: 1
Sample path length: 75
Saved: kdtree_edge.json
Time: 10.11436128616333
